In [5]:
import tkinter as tk
from tkinter import messagebox
import json
import os
from datetime import datetime


# CONFIG


DATA_FILE = "tn_evm_votes.json"
TOTAL_SEATS = 234

PARTIES = [
    ("TVK", "Temple"),
    ("DMDK", "Drums"),
    ("CONG", "Hand"),
    ("DMK", "Rising Sun"),
    ("BJP", "Lotus"),
    ("AIADMK", "Two Leaves"),
    ("NTK", "Farmer")
]



# ELECTION SYSTEM


class ElectionSystem:
    def __init__(self):
        self.voters = []
        self.voted_ids = set()

        self.party_votes = {
            party: 0
            for party, symbol in PARTIES
        }

        self.load_data()

    def load_data(self):

        if not os.path.exists(DATA_FILE):
            return

        try:
            with open(DATA_FILE, "r") as f:
                data = json.load(f)

            self.voters = data.get("voters", [])

            for voter in self.voters:

                self.voted_ids.add(
                    voter["voter_id"]
                )

                party = voter["party"]

                if party in self.party_votes:
                    self.party_votes[party] += 1

        except:
            pass

    def save_data(self):

        with open(DATA_FILE, "w") as f:
            json.dump(
                {"voters": self.voters},
                f,
                indent=4
            )

    def cast_vote(self, name, voter_id, party):

        if voter_id in self.voted_ids:
            return False, "This voter already voted."

        vote = {
            "name": name,
            "voter_id": voter_id,
            "party": party,
            "timestamp": str(datetime.now())
        }

        self.voters.append(vote)

        self.voted_ids.add(voter_id)

        self.party_votes[party] += 1

        self.save_data()

        return True, "Vote Recorded Successfully"

    def get_results(self):

        total_votes = sum(self.party_votes.values())

        result = []

        for party, votes in self.party_votes.items():

            seats = 0

            if total_votes > 0:
                seats = int(
                    votes / total_votes * TOTAL_SEATS
                )

            result.append(
                (party, votes, seats)
            )

        result.sort(
            key=lambda x: x[1],
            reverse=True
        )

        return result



# GUI


class EVM:
    def __init__(self, root):

        self.root = root

        self.root.title(
            "Tamil Nadu Legislative Assembly Election"
        )

        self.root.geometry("1100x750")
        self.root.configure(bg="#8D99AE")

        self.system = ElectionSystem()

        self.selected_voter_name = ""
        self.selected_voter_id = ""

        self.build_login_screen()

    
    # LOGIN SCREEN
    

    def build_login_screen(self):

        self.clear_window()

        frame = tk.Frame(
            self.root,
            bg="white",
            padx=30,
            pady=30
        )

        frame.place(
            relx=0.5,
            rely=0.5,
            anchor="center"
        )

        tk.Label(
            frame,
            text="ELECTRONIC VOTING MACHINE",
            font=("Arial", 22, "bold"),
            bg="white"
        ).pack(pady=10)

        tk.Label(
            frame,
            text="Full Name",
            bg="white"
        ).pack()

        self.name_entry = tk.Entry(
            frame,
            width=35,
            font=("Arial", 12)
        )
        self.name_entry.pack(pady=5)

        tk.Label(
            frame,
            text="12 Character Voter ID",
            bg="white"
        ).pack()

        self.id_entry = tk.Entry(
            frame,
            width=35,
            font=("Arial", 12)
        )
        self.id_entry.pack(pady=5)

        tk.Button(
            frame,
            text="ENTER EVM",
            bg="#1976D2",
            fg="white",
            font=("Arial", 12, "bold"),
            command=self.validate_voter
        ).pack(pady=20)

    def validate_voter(self):

        name = self.name_entry.get().strip()
        voter_id = self.id_entry.get().strip().upper()

        if not name:
            messagebox.showerror(
                "Error",
                "Enter Name"
            )
            return

        if len(voter_id) != 12:
            messagebox.showerror(
                "Error",
                "Voter ID must be 12 characters"
            )
            return

        if voter_id in self.system.voted_ids:
            messagebox.showerror(
                "Blocked",
                "This voter already voted"
            )
            return

        self.selected_voter_name = name
        self.selected_voter_id = voter_id

        self.build_evm_screen()

    
    # EVM SCREEN
   

    def build_evm_screen(self):

        self.clear_window()

        machine = tk.Frame(
            self.root,
            bg="#CFD8DC",
            bd=10,
            relief="ridge"
        )

        machine.pack(
            fill="both",
            expand=True,
            padx=30,
            pady=30
        )

        title = tk.Label(
            machine,
            text="TAMIL NADU LEGISLATIVE ASSEMBLY ELECTION",
            font=("Arial", 20, "bold"),
            bg="#455A64",
            fg="white",
            pady=15
        )

        title.pack(fill="x")

        body = tk.Frame(
            machine,
            bg="#ECEFF1"
        )

        body.pack(
            fill="both",
            expand=True,
            padx=20,
            pady=20
        )

        self.status_light = tk.Label(
            machine,
            text="READY",
            bg="green",
            fg="white",
            font=("Arial", 14, "bold"),
            width=20
        )

        self.status_light.pack(
            pady=10
        )

        for i, (party, symbol) in enumerate(PARTIES):

            row = tk.Frame(
                body,
                bg="white",
                bd=2,
                relief="solid"
            )

            row.pack(
                fill="x",
                pady=4
            )

            tk.Label(
                row,
                text=f"{i+1}",
                width=4,
                font=("Arial", 14, "bold"),
                bg="white"
            ).pack(
                side="left",
                padx=5
            )

            tk.Label(
                row,
                text=party,
                width=20,
                anchor="w",
                font=("Arial", 14),
                bg="white"
            ).pack(
                side="left"
            )

            tk.Label(
                row,
                text=symbol,
                width=20,
                anchor="w",
                font=("Arial", 12),
                bg="white"
            ).pack(
                side="left"
            )

            vote_btn = tk.Button(
                row,
                text="●",
                bg="#1565C0",
                fg="white",
                font=("Arial", 18, "bold"),
                width=4,
                height=1,
                command=lambda p=party:
                self.vote(p)
            )

            vote_btn.pack(
                side="right",
                padx=20,
                pady=5
            )

        bottom = tk.Frame(
            machine,
            bg="#CFD8DC"
        )

        bottom.pack(
            fill="x",
            pady=10
        )

        tk.Button(
            bottom,
            text="SHOW RESULTS",
            bg="orange",
            font=("Arial", 12, "bold"),
            command=self.show_results
        ).pack(
            side="left",
            padx=10
        )

        tk.Button(
            bottom,
            text="NEW VOTER",
            bg="green",
            fg="white",
            font=("Arial", 12, "bold"),
            command=self.build_login_screen
        ).pack(
            side="left"
        )

    
    # VOTE
    

    def vote(self, party):

        ok, msg = self.system.cast_vote(
            self.selected_voter_name,
            self.selected_voter_id,
            party
        )

        if not ok:
            messagebox.showerror(
                "Error",
                msg
            )
            return

        self.root.bell()

        self.status_light.config(
            text="VOTE CAST ✓",
            bg="red"
        )

        messagebox.showinfo(
            "Success",
            f"Vote Cast For {party}"
        )

        self.build_login_screen()

    
    # RESULTS
   

    def show_results(self):

        win = tk.Toplevel(self.root)

        win.title("Election Results")

        win.geometry("600x450")

        text = tk.Text(
            win,
            font=("Consolas", 12)
        )

        text.pack(
            fill="both",
            expand=True
        )

        text.insert(
            "end",
            "TAMIL NADU ELECTION RESULTS\n\n"
        )

        text.insert(
            "end",
            f"{'PARTY':<15}{'VOTES':<10}{'SEATS'}\n"
        )

        text.insert(
            "end",
            "-" * 40 + "\n"
        )

        for party, votes, seats in self.system.get_results():

            text.insert(
                "end",
                f"{party:<15}{votes:<10}{seats}\n"
            )

    
    # HELPERS
    

    def clear_window(self):

        for widget in self.root.winfo_children():
            widget.destroy()



# MAIN


root = tk.Tk()

app = EVM(root)

root.mainloop()